# Phase 1 Task B — ResNet-50 × 3 Seeds

**Model:** ResNet-50
**Seeds:** 42, 123, 456
**Batch:** 64
**Output:** wandb `woodbench-taskB`

Chạy notebook này trên 1 GPU (vast.ai).

In [ ]:
from pathlib import Path
NOTEBOOK = Path("__file__").resolve()
ROOT = NOTEBOOK
for _ in range(10):
    ROOT = ROOT.parent
    if (ROOT / "pyproject.toml").exists() or (ROOT / ".git").exists(): break
import sys, os
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print(f"[ROOT] {ROOT}")

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path := ROOT / ".env")
print(f"[.env] Loaded from {dotenv_path}")

import subprocess, sys
for pkg in ["timm", "albumentations", "scikit-learn", "seaborn", "wandb", "ptflops"]:
    try: __import__(pkg.replace("-", "_"))
    except: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

os.environ.setdefault("WANDB_API_KEY", os.getenv("WANDB_API_KEY", ""))
print(f"[wandb] API key: {'set' if os.getenv('WANDB_API_KEY') else 'MISSING'}")

In [ ]:
from src.data import get_dataloaders
DATA_ROOT = ROOT / "data" / "task-B"
tl, vl, testl = get_dataloaders(str(DATA_ROOT), batch_size=64, num_workers=4, task="taskB")
print(f"[Data] Train={len(tl.dataset)}, Val={len(vl.dataset)}, Test={len(testl.dataset)}")
print(f"[Data] Classes={len(tl.dataset.classes)}")

In [ ]:
import wandb, torch, numpy as np
from src.models import create_model
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_loss_function
from src.utils import set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Device] {DEVICE}")

In [ ]:
def train_one(backbone, seed, data_root, batch_size=64, epochs=50):
    set_seed(seed)
    output_dir = ROOT / "experiments" / "taskB" / backbone / f"seed_{seed}"
    output_dir.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader, _ = get_dataloaders(
        data_root=data_root, batch_size=batch_size, num_workers=4, task="taskB")
    num_classes = len(train_loader.dataset.classes)
    print(f"[{backbone} s{seed}] train={len(train_loader.dataset)}, classes={num_classes}")

    run = wandb.init(project="woodbench-taskB", name=f"{backbone}_s{seed}",
        config={"backbone": backbone, "seed": seed, "epochs": epochs,
                "batch_size": batch_size, "num_classes": num_classes,
                "lr_head": 1e-4, "lr_backbone": 1e-5,
                "label_smoothing": 0.1, "scheduler": "cosine", "amp": True},
        dir=str(output_dir))

    model = create_model(backbone, num_classes=num_classes, pretrained=True)
    optimizer = create_optimizer(model, lr=1e-4, weight_decay=1e-4, backbone_lr=1e-5)
    scheduler = create_scheduler(optimizer, epochs=epochs, scheduler_type="cosine")
    criterion = create_loss_function("label_smoothing", label_smoothing=0.1)

    engine = TrainingEngine(model=model, train_loader=train_loader, val_loader=val_loader,
        optimizer=optimizer, scheduler=scheduler, criterion=criterion,
        device=DEVICE, use_amp=True)
    result = engine.fit(epochs=epochs, output_dir=output_dir,
                       name=f"{backbone}_seed{seed}", save_best_only=True)
    run.finish()
    print(f"[{backbone} s{seed}] val_acc={result['best_val_acc']:.2f}%")
    return result

In [ ]:
SEEDS = [42, 123, 456]
results = {}

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"ResNet-50 | seed={seed} ({SEEDS.index(seed)+1}/{len(SEEDS)})")
    print(f"{'='*60}")
    results[seed] = train_one("resnet50", seed, str(DATA_ROOT), batch_size=64, epochs=50)

print("\n" + "="*60)
print("ResNet-50 — Summary")
for seed, res in results.items():
    print(f"  seed={seed}: {res['best_val_acc']:.2f}%")
accs = [r['best_val_acc'] for r in results.values()]
print(f"  Mean: {np.mean(accs):.2f}% ± {np.std(accs):.2f}%")